# 02 · Build, evaluate & optimize the agent

This is the heart of the workshop. You'll build **TrailMate** — a grounded gear expert — then **hill climb** in two stages: first by hand, one lever at a time, then **automatically** with Agent Optimizer.

```mermaid
flowchart LR
    A["v1 · Basecamp<br/>frontier GPT + simple instructions"] -->|"lever 1: instructions"| B["v2 · Clearpath<br/>optimized instructions"]
    B -->|"lever 2: model"| C["v3 · Trailfinder<br/>Model Router"]
    C -->|"promote strongest manual version"| D(("new baseline"))
    D -->|"Agent Optimizer (portal)"| F["v4 · Summit<br/>auto-optimized"]
    A -.measure.-> E[(frozen rubric<br/>+ test set)]
    B -.measure.-> E
    C -.measure.-> E
    F -.measure.-> E
```

An **agent** here is simply a **model + instructions + a tool**. TrailMate's tool is **file-search** over 10 Contoso product manuals, so it answers from real specs instead of guessing.

### Learning objectives
By the end of this notebook you'll be able to:
- **Build** a grounded prompt agent (vector store + file-search + instructions).
- **Observe** its behavior with traces and run metrics.
- **Measure** a baseline with a frozen rubric evaluator — your "altitude."
- **Manually climb** one lever at a time: optimize instructions (v2), then optimize the model (v3).
- **Promote** the strongest manual version as a new baseline for automated optimization.
- **Automate the climb** with Agent Optimizer (portal) and compare its best candidate (v4) against your manual best.

> ⏱️ **~45 minutes** · Steps 1–8 run entirely in this notebook; step 9 (Agent Optimizer) is a short portal detour. The rubric and test set stay **frozen** across every version so every score is directly comparable.


## 1 · Set up and locate the workshop data

Same `.env` as before. We also point at the assets that make TrailMate grounded and measurable: the **10 product manuals**, the **v1 / optimized instructions**, and the **frozen test set**.


In [1]:
import os
import time
import uuid
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.core.pipeline.policies import SansIOHTTPPolicy


def find_agent_builder() -> Path:
    """Locate the foundry/agent-builder folder from anywhere in the tree."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Could not locate foundry/agent-builder.")


class EnsureEvalPreviewHeader(SansIOHTTPPolicy):
    """Add the Evaluations preview opt-in to EVERY request (incl. LRO polling).

    The evaluators API is preview and needs `Foundry-Features: Evaluations=V1Preview`.
    The SDK adds it to the initial call but not to the poller's follow-up GETs, so
    we ensure it here — merging with any features the SDK already set.
    """

    def on_request(self, request):
        headers = request.http_request.headers
        current = headers.get("Foundry-Features")
        if not current:
            headers["Foundry-Features"] = "Evaluations=V1Preview"
        elif "Evaluations=" not in current:
            headers["Foundry-Features"] = current + ",Evaluations=V1Preview"


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")

# Clients: project SDK for agents/evals, OpenAI-compatible for calls + eval runs.
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
    per_call_policies=[EnsureEvalPreviewHeader()],
)
openai_client = project_client.get_openai_client()

# Names + models for the three versions we'll create.
AGENT_NAME = os.environ.get("TRAILMATE_AGENT_NAME", "trailmate")
FRONTIER_MODEL = os.environ.get("TRAILMATE_MODEL_DEPLOYMENT", "gpt-5.4")
ROUTER_MODEL = "model-router"

# The assets that make TrailMate grounded and measurable.
MANUALS_DIR = AB / "src" / "data" / "manuals"
EVAL_CASES = AB / "src" / "data" / "evaluation-cases.jsonl"
INSTRUCTIONS_V1 = AB / "src" / "agent" / "instructions.md"
INSTRUCTIONS_OPTIMIZED = AB / "src" / "agent" / "instructions_optimized.md"

print(f"Agent: {AGENT_NAME}  |  frontier: {FRONTIER_MODEL}  |  router: {ROUTER_MODEL}")
print(f"Manuals: {len(list(MANUALS_DIR.glob('*.md')))} files  |  test set: {EVAL_CASES.name}")


Agent: trailmate  |  frontier: gpt-5.4  |  router: model-router
Manuals: 10 files  |  test set: evaluation-cases.jsonl


## 2 · Build TrailMate v1 (Basecamp)

Three moves make an agent:
1. **Vector store** — index the 10 manuals so they're searchable by meaning.
2. **File-search tool** — lets the agent retrieve the right manual at answer time.
3. **Prompt agent** — a frontier model + a **deliberately simple** set of instructions.

That last choice is on purpose: v1's instructions are vague, which leaves **headroom** we'll measure and then close. This is *Basecamp* — capable engine, weak guidance.

> 💡 **Warm up tracing *and* evaluations (recommended):** After the cell below creates the agent, open it in the Foundry portal — **Agents → `trailmate` → Playground** — and do this **before** asking anything:
> 1. Open the **Metrics** (evaluations) panel and **turn on all** the run metrics/evaluators (quality, safety, etc.). This makes each run emit both **traces** *and* **eval scores**.
> 2. Now ask one quick question like *"What can you do?"*.
>
> That first interaction activates the pipelines, so the runs you send from section 3 onward show up in the portal with metrics much sooner. Then come back here.

<!-- TODO: screenshot — Playground with the Metrics/evaluations panel expanded and all toggles on → assets/02-00-playground-metrics-on.png -->


In [2]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

VECTOR_STORE_NAME = "trailmate-manuals"


def get_or_create_vector_store():
    """Reuse the manuals vector store if it exists; upload only once."""
    for store in openai_client.vector_stores.list():
        if getattr(store, "name", None) == VECTOR_STORE_NAME:
            print(f"Reusing vector store {store.id} — manuals already uploaded.")
            return store
    print("Creating vector store and uploading the 10 manuals…")
    store = openai_client.vector_stores.create(name=VECTOR_STORE_NAME)
    streams = [p.open("rb") for p in sorted(MANUALS_DIR.glob("*.md"))]
    try:
        openai_client.vector_stores.file_batches.upload_and_poll(
            vector_store_id=store.id, files=streams,
        )
    finally:
        for s in streams:
            s.close()
    print(f"  Uploaded manuals → {store.id}")
    return store


vector_store = get_or_create_vector_store()

# v1 = frontier model + file-search + deliberately simple instructions.
v1_definition = PromptAgentDefinition(
    model=FRONTIER_MODEL,
    instructions=INSTRUCTIONS_V1.read_text(encoding="utf-8"),
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)
v1 = project_client.agents.create_version(AGENT_NAME, definition=v1_definition)
print(f"\nTrailMate v1 (Basecamp) ready → '{AGENT_NAME}' v{getattr(v1, 'version', '?')} on {FRONTIER_MODEL}")


Creating vector store and uploading the 10 manuals…
  Uploaded manuals → vs_Ew5WxHpor25b7czWw8SY6m7x

TrailMate v1 (Basecamp) ready → 'trailmate' v1 on gpt-5.4


## 3 · Test and observe

Let's ask TrailMate a real question — one that needs it to retrieve and compare **two** tent manuals. Read the answer, but also notice *that this is observable*: every run emits a **trace** (the file-search tool call, the model, the steps) and **metrics** (tokens, latency, quality). That's **AgentOps** — you don't just see *what* it answered, but *whether the answer was any good*.

> ⏳ **Give it a moment:** Traces and metrics can take a **short delay** (often up to a minute or two) to appear in the portal after a run — and the *very first* trace for a new agent can take longest. If you warmed up tracing at the end of section 2 (asked the agent a question in the Playground), your runs here will show up much faster. Re-open the **Traces** tab after a minute if it looks empty at first.

<!-- TODO: screenshot — Foundry portal → Agents → trailmate → Traces, showing the file_search span + run metrics → assets/02-01-trace-metrics.png -->


In [3]:
# Invoke the agent by reference through the OpenAI-compatible responses API.
question = (
    "I'm expecting heavy rain. Which is more weatherproof, the Alpine Explorer "
    "Tent or the TrailMaster X4 Tent?"
)
response = openai_client.responses.create(
    model=FRONTIER_MODEL,
    input=question,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
)
print(response.output_text)
print("\n👉 Open this run in the Foundry portal (Agents → Traces) to see the")
print("   file_search tool call, the model, tokens, latency, and quality score.")


For heavy rain, the Alpine Explorer Tent looks more weatherproof.

The biggest reason is its rainfly waterproof rating: the Alpine Explorer is rated at 3000mm, while the TrailMaster X4 is rated at 2000mm. Both are listed as waterproof, both include a rainfly, and both are 3-season tents, but the Alpine’s higher rainfly rating suggests better rain protection in sustained wet conditions  

Quick comparison:
- Alpine Explorer Tent: waterproof, rainfly included, 3000mm rainfly rating 
- TrailMaster X4 Tent: waterproof, rainfly included, 2000mm rainfly rating 

One thing to keep in mind: the Alpine is much larger and heavier at 17 lbs, while the TrailMaster X4 is 12 lbs, so the TrailMaster may still be the better pick if portability matters more than maximum rain protection  

If you want, I can also compare them for wind resistance, space, and ease of setup.

👉 Open this run in the Foundry portal (Agents → Traces) to see the
   file_search tool call, the model, tokens, latency, and quality

## 4 · Set up the measurement (test set + frozen rubric)

You can't say a version is *better* until you can measure *good*. Our yardstick has two frozen parts, used identically for v1, v2, and v3:

- **A test set** — [`evaluation-cases.jsonl`](../../src/data/evaluation-cases.jsonl): 14 queries spanning the failure modes v1 should struggle with (grounding, refusal, wrong-product traps, off-topic).
- **A rubric evaluator** — an LLM judge scoring each answer on grounding & citation, correctness, appropriate refusal, and tone. Our reference rubric is [`trailmate-quality.yaml`](../../src/data/evaluators/trailmate-quality.yaml); here we register it as a Foundry evaluator and **reuse the same one every time** so scores are comparable.

We create these **once**, then define a small `run_eval(...)` helper.

> 🛠️ **New-Foundry SDK note:** this uses the `azure-ai-projects` evaluation flow (`evals.create` + `evals.runs.create`). If a field name differs in your installed version, the [Evaluate your AI agents](https://learn.microsoft.com/azure/foundry/observability/how-to/evaluate-agent) quickstart is the source of truth — or run the evaluation from the portal (**Evaluations** tab) as the fallback.

> ⚠️ **Watch for "Partial" runs:** the portal's pass-rate % only counts items the judge could actually score — errored items are silently excluded. That can make a version look great while it was really graded on a shrinking slice of the frozen set, breaking the apples-to-apples comparison. We map the judge's input to `output_text` (the final answer) rather than `output_items` (the full raw response incl. file-search payloads) to keep judge inputs small and reliable. Section 8's summary table surfaces `errored` explicitly so this is never hidden.


In [4]:
from azure.core.exceptions import ResourceExistsError
from azure.ai.projects.models import (
    AgentEvaluatorGenerationJobSource,
    EvaluatorGenerationInputs,
    EvaluatorGenerationJob,
    TestingCriterionAzureAIEvaluator,
)
from openai.types.eval_create_params import DataSourceConfigCustom

# 1. Upload the frozen test set as a versioned dataset (idempotent: reuse if it
#    already exists from a previous run of this notebook).
DATASET_NAME, DATASET_VERSION = "trailmate-eval-cases", "1"
try:
    dataset = project_client.datasets.upload_file(
        name=DATASET_NAME, version=DATASET_VERSION, file_path=str(EVAL_CASES),
    )
    print(f"Dataset uploaded: {dataset.id}")
except ResourceExistsError:
    dataset = project_client.datasets.get(name=DATASET_NAME, version=DATASET_VERSION)
    print(f"Dataset already exists — reusing: {dataset.id}")

# 2. Create ONE rubric evaluator and reuse it for every version (frozen yardstick).
#    It scores the 4 dimensions in trailmate-quality.yaml against each case.
gen_job = EvaluatorGenerationJob(
    inputs=EvaluatorGenerationInputs(
        model=FRONTIER_MODEL,
        evaluator_name=f"trailmate-quality-{uuid.uuid4().hex[:8]}",
        evaluator_display_name="TrailMate Quality",
        sources=[AgentEvaluatorGenerationJobSource(agent_name=AGENT_NAME)],
    ),
)
poller = project_client.beta.evaluators.begin_create_generation_job(job=gen_job)
while not poller.done():
    time.sleep(10)
rubric = poller.result()
print(f"Rubric evaluator ready: {rubric.name} v{rubric.version}")

# 3. Define testing criteria (the rubric) + an evaluation container (fixed schema).
#    IMPORTANT: map response to output_TEXT (the final answer), not output_items
#    (the full raw response incl. file-search tool-call payloads). output_items
#    can balloon to tens of thousands of tokens per item and cause the judge call
#    to error out instead of score — output_text is the small, sufficient input
#    our rubric (grounding/correctness/refusal/tone) actually needs.
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="TrailMate Quality",
        evaluator_name=rubric.name,
        initialization_parameters={"deployment_name": FRONTIER_MODEL},
        data_mapping={"query": "{{item.query}}", "response": "{{sample.output_text}}"},
    ),
]
evaluation = openai_client.evals.create(
    name="TrailMate Quality Evaluation",
    data_source_config=DataSourceConfigCustom(
        type="custom",
        item_schema={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        include_sample_schema=True,
    ),
    testing_criteria=testing_criteria,
)
print(f"Evaluation container ready: {evaluation.id}")


Dataset uploaded: azureai://accounts/foundry-iziws3ekpbw2a/projects/foundry-workshop-iziws3ekpbw2a/data/trailmate-eval-cases/versions/1
Rubric evaluator ready: trailmate-quality-8599b335 v1
Evaluation container ready: eval_d2868259c9ab4d91af22c5ee63fe8947


## 5 · Baseline eval — read your altitude

Now run the evaluation against **v1**. The service sends every test query to the agent, captures the response, and scores it with our rubric. The result is your **baseline altitude** — the number every later change must beat. We define a reusable `run_eval(...)` helper here, then call it for v1.


In [5]:
def run_eval(label: str, agent_version=None):
    """Run the frozen evaluation against one agent version, timing it and
    capturing token usage — the signals we need to show real progression."""
    target = {"type": "azure_ai_agent", "name": AGENT_NAME}
    if agent_version is not None:
        target["version"] = str(agent_version)

    start = time.perf_counter()
    run = openai_client.evals.runs.create(
        eval_id=evaluation.id,
        name=f"TrailMate {label}",
        data_source={
            "type": "azure_ai_target_completions",
            "source": {"type": "file_id", "id": dataset.id},
            "input_messages": {
                "type": "template",
                "template": [{"type": "message", "role": "user",
                              "content": {"type": "input_text", "text": "{{item.query}}"}}],
            },
            "target": target,
        },
    )
    print(f"  {label}: run {run.id} started — polling…")
    while run.status not in ("completed", "failed"):
        time.sleep(5)
        run = openai_client.evals.runs.retrieve(run_id=run.id, eval_id=evaluation.id)
    elapsed_s = round(time.perf_counter() - start, 1)

    # per_model_usage totals tokens per model actually served — for the router
    # (v2/v3) this can list MULTIPLE models, showing its per-query choices.
    usage = run.per_model_usage or []
    total_tokens = sum(getattr(u, "total_tokens", 0) or 0 for u in usage)
    models_used = sorted({getattr(u, "run_model_name", "?") for u in usage}) or ["?"]

    print(f"  {label}: {run.status}  |  {run.result_counts}  |  "
          f"{elapsed_s}s (incl. polling)  |  {total_tokens} tokens  |  models: {', '.join(models_used)}")
    return {"run": run, "elapsed_s": elapsed_s, "total_tokens": total_tokens, "models_used": models_used}


# Baseline: measure v1.
baseline_run = run_eval("v1 · Basecamp", getattr(v1, "version", None))
print("\nReport:", getattr(baseline_run["run"], "report_url", "(Foundry portal → Evaluations)"))


  v1 · Basecamp: run evalrun_048fd8b152d04c46ae7ec81aad4bb9a9 started — polling…
  v1 · Basecamp: completed  |  ResultCounts(errored=0, failed=2, passed=12, total=14, skipped=0)  |  301.2s (incl. polling)  |  577142 tokens  |  models: azure_ai_agent_target, azure_ai_evaluation, gpt-5.4

Report: https://ai.azure.com/nextgen/r/uQtrTprHTB2ET1hEQ5URvA,rg-model-masterylod65270515,,foundry-iziws3ekpbw2a,foundry-workshop-iziws3ekpbw2a/build/evaluations/eval_d2868259c9ab4d91af22c5ee63fe8947/run/evalrun_048fd8b152d04c46ae7ec81aad4bb9a9


In [6]:
# Diagnostic: inspect WHY items errored/failed (not just the pass/fail counts).
def show_eval_errors(run, limit: int = 5):
    """Print the query + error/status for each non-passing item in a run."""
    items = openai_client.evals.runs.output_items.list(run_id=run.id, eval_id=evaluation.id)
    shown = 0
    for item in items:
        status = getattr(item, "status", None)
        if status == "pass":
            continue
        query = None
        try:
            query = item.datasource_item.get("query")
        except Exception:
            pass
        print(f"--- item {getattr(item, 'id', '?')}  status={status}  query={query!r}")
        for result in getattr(item, "results", []) or []:
            print(f"    result={getattr(result, 'result', None)}  "
                  f"reason={getattr(result, 'reason', None)}  "
                  f"error={getattr(result, 'error', None) or getattr(result, 'sample', None)}")
        shown += 1
        if shown >= limit:
            print(f"... (showing first {limit} non-passing items)")
            break
    if shown == 0:
        print("No failed/errored items found (or output_items API shape differs — check the portal report instead).")


print("== v1 · Basecamp errors/failures ==")
show_eval_errors(baseline_run["run"])


== v1 · Basecamp errors/failures ==
--- item 1  status=completed  query='How waterproof is the TrailMaster X4 Tent?'
    result=None  reason=The verdict is driven most by grounded_factual_accuracy (2) and uncertainty_limitation_acknowledgment (1): the assistant confidently states a 2000 mm rainfly rating, 3-season use, and customer storm performance even though no such product facts are visible in the conversation. It still scores better on task_relevance_and_completion (4) and clarity_and_customer_facing_communication (4) because it directly answers the waterproofing question and explains the practical takeaway clearly. Overall, general_quality is only 2 because the response is polished and relevant but not reliably grounded in the information provided.  error={'usage': {'prompt_tokens': 2188, 'completion_tokens': 580, 'total_tokens': 2768, 'cached_tokens': 0}, 'model': 'gpt-5.4'}
--- item 2  status=completed  query="I'm expecting heavy rain. Which is more weatherproof, the Alpine Exp

## 6 · Lever 1 — Optimize the instructions (v2 Clearpath)

Now the first climb. We change **exactly one thing**: the instructions. v2 keeps v1's model and the same file-search over the same vector store, but swaps the vague v1 instructions for the **optimized** set — search first and cite the product, refuse when the manuals don't cover it, disambiguate similar products, stay in scope.

The question this answers: *does clearer guidance raise quality on the same engine?* Re-run the **same** eval to find out.


In [7]:
# v2: same model, same tool + vector store — ONLY the instructions change.
v2_definition = PromptAgentDefinition(
    model=FRONTIER_MODEL,
    instructions=INSTRUCTIONS_OPTIMIZED.read_text(encoding="utf-8"),
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)
v2 = project_client.agents.create_version(AGENT_NAME, definition=v2_definition)
print(f"TrailMate v2 (Clearpath) → version {getattr(v2, 'version', '?')} on {FRONTIER_MODEL}")

# Same frozen eval, new version.
v2_run = run_eval("v2 · Clearpath", getattr(v2, "version", None))


TrailMate v2 (Clearpath) → version 2 on gpt-5.4
  v2 · Clearpath: run evalrun_25d995eb9fd44171bf5ccb75dafbdd7f started — polling…
  v2 · Clearpath: completed  |  ResultCounts(errored=0, failed=3, passed=11, total=14, skipped=0)  |  358.8s (incl. polling)  |  581006 tokens  |  models: azure_ai_agent_target, azure_ai_evaluation, gpt-5.4


## 7 · Lever 2 — Optimize the model (v3 Trailfinder)

Second climb, second lever: the **model**. We keep v2's optimized instructions and swap the model from `gpt-5.4` to **`model-router`**. Because only the model changed, any shift in cost, latency, or quality is *attributable to the model* — the discipline of one lever at a time.

The question this answers: *can a per-request model choice cut cost and latency while holding the quality v2 already earned?* Re-run the **same** eval to find out.


In [8]:
# v3: same optimized instructions as v2, same tool + vector store — ONLY the
# model changes (frontier → router).
v3_definition = PromptAgentDefinition(
    model=ROUTER_MODEL,
    instructions=INSTRUCTIONS_OPTIMIZED.read_text(encoding="utf-8"),
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)
v3 = project_client.agents.create_version(AGENT_NAME, definition=v3_definition)
print(f"TrailMate v3 (Trailfinder) → version {getattr(v3, 'version', '?')} on {ROUTER_MODEL}")

# Same frozen eval, once more.
v3_run = run_eval("v3 · Trailfinder", getattr(v3, "version", None))


TrailMate v3 (Trailfinder) → version 3 on model-router
  v3 · Trailfinder: run evalrun_14f5fa382f4b4babb01a444d98554cab started — polling…
  v3 · Trailfinder: completed  |  ResultCounts(errored=0, failed=5, passed=9, total=14, skipped=0)  |  235.4s (incl. polling)  |  166266 tokens  |  models: azure_ai_agent_target, azure_ai_evaluation, gpt-5.4, gpt-5.6-luna-2026-07-09


## 8 · Compare and promote a new baseline

Three versions, one frozen yardstick. This is where the manual hill climb usually plateaus:

- **v1 → v2** (instructions) should **raise quality** — clearer guidance, same engine.
- **v2 → v3** (model) should **cut cost and latency** while *holding* quality — not necessarily improving it further.

That's the manual ceiling: you can trade cost/latency for a model, but a human iterating on instructions by hand only goes so far. Let's pick the strongest version as the **new baseline**, then hand the climb to **Agent Optimizer** to search further automatically.


In [9]:
import pandas as pd


def counts(entry):
    """Read pass/fail/error/elapsed/tokens from a run_eval() result.

    We surface `errored` explicitly: the portal's pass-rate % only counts GRADED
    items, silently excluding errored ones — which would make versions look
    great while secretly grading a shrinking, uneven slice of the frozen set.
    """
    rc = entry["run"].result_counts
    get = (lambda k: rc.get(k)) if isinstance(rc, dict) else (lambda k: getattr(rc, k, None))
    passed, failed, errored, total = get("passed"), get("failed"), get("errored"), get("total")
    graded = (passed or 0) + (failed or 0)
    rate = round(passed / graded, 2) if graded else None
    return {
        "passed": passed,
        "failed": failed,
        "errored": errored,
        "graded_of_total": f"{graded}/{total}",
        "pass_rate (of graded)": rate,
        "elapsed_s": entry["elapsed_s"],
        "total_tokens": entry["total_tokens"],
        "models_used": ", ".join(entry["models_used"]),
    }


# One row per manual version; keep the agent_version number so we can promote
# the winner by reference (used to target Agent Optimizer in step 9).
rows = [
    {"version": "v1 · Basecamp", "agent_version": getattr(v1, "version", None),
     "model": FRONTIER_MODEL, "lever": "starting point", **counts(baseline_run)},
    {"version": "v2 · Clearpath", "agent_version": getattr(v2, "version", None),
     "model": FRONTIER_MODEL, "lever": "instructions → optimized", **counts(v2_run)},
    {"version": "v3 · Trailfinder", "agent_version": getattr(v3, "version", None),
     "model": ROUTER_MODEL, "lever": "model → router", **counts(v3_run)},
]
summary = pd.DataFrame(rows)

# Deltas vs the v1 baseline make "did it actually improve?" concrete at a glance.
summary["Δ pass_rate vs v1"] = (summary["pass_rate (of graded)"] - summary.loc[0, "pass_rate (of graded)"]).round(2)
summary["Δ tokens vs v1"] = summary["total_tokens"] - summary.loc[0, "total_tokens"]

if (summary["errored"].fillna(0) > 0).any():
    print("⚠️  At least one version has errored items — it was graded on FEWER than")
    print("    the full frozen set. Fix errors before trusting this comparison; run")
    print("    show_eval_errors(entry['run']) on the affected version to see why.\n")

# Promote the strongest MANUAL version as the new baseline: highest pass_rate
# first, then fewer tokens, then lower latency as tie-breakers.
ranked = summary.sort_values(
    by=["pass_rate (of graded)", "total_tokens", "elapsed_s"],
    ascending=[False, True, True],
)
promoted = ranked.iloc[0]
PROMOTED_LABEL = promoted["version"]
PROMOTED_AGENT_VERSION = promoted["agent_version"]

print(f"🏁 Promoted baseline for automated optimization: {PROMOTED_LABEL} "
      f"(agent version {PROMOTED_AGENT_VERSION})\n")
summary


🏁 Promoted baseline for automated optimization: v1 · Basecamp (agent version 1)



,version,agent_version,model,lever,passed,failed,errored,graded_of_total,pass_rate (of graded),elapsed_s,total_tokens,models_used,Δ pass_rate vs v1,Δ tokens vs v1
0,v1 · Basecamp,1,gpt-5.4,starting point,12,2,0,14/14,0.86,301.2,577142,"azure_ai_agent_target, azure_ai_evaluation, gp...",0.00,0
1,v2 · Clearpath,2,gpt-5.4,instructions → optimized,11,3,0,14/14,0.79,358.8,581006,"azure_ai_agent_target, azure_ai_evaluation, gp...",-0.07,3864
2,v3 · Trailfinder,3,model-router,model → router,9,5,0,14/14,0.64,235.4,166266,"azure_ai_agent_target, azure_ai_evaluation, gp...",-0.22,-410876


## 9 · Automate the climb — Agent Optimizer (v4 Summit)

Manual hill climbing has a ceiling: you tried two levers by hand and promoted the best result — printed above — as your new baseline. Now hand the climb to **Agent Optimizer**. Point it at that baseline in the Foundry portal and it generates and scores **multiple candidate instruction sets** automatically, using the exact same frozen rubric and test set you've been using here.

1. In the Foundry portal, open **Agents → `trailmate` → Optimize**.
2. Set the **starting point** to the promoted baseline version number printed in step 8.
3. Run the optimizer, review its candidates and their scores, then **apply the best candidate** as a new agent version — this becomes **v4 · Summit**.

<!-- TODO: screenshot — Foundry portal → Agents → Optimize, showing candidate instructions + eval score → assets/02-02-agent-optimizer.png -->

> 🔎 **Why this matters:** the manual climb showed you *what* one-lever-at-a-time optimization looks like. Agent Optimizer runs that same discipline **at scale** — many candidates, scored automatically, faster than you could iterate by hand.

Once you've applied a candidate in the portal, note its **version number** and run the cell below to measure it with the exact same yardstick.


In [10]:
from IPython.display import display

# Fill in the agent version number Agent Optimizer created after you applied
# its best candidate in the portal (Agents → trailmate → Versions).
V4_AGENT_VERSION = None  # e.g. 4

if V4_AGENT_VERSION is None:
    print("Set V4_AGENT_VERSION above to the version number you applied in the portal, then re-run this cell.")
else:
    v4_run = run_eval("v4 · Summit", V4_AGENT_VERSION)

    # Look up what model the optimizer actually landed on (it may keep the
    # router, or the Optional exercise in step 9's callout may have explored
    # model candidates too).
    v4_details = project_client.agents.get_version(AGENT_NAME, V4_AGENT_VERSION)
    v4_model = getattr(v4_details.definition, "model", "?")

    summary.loc[len(summary)] = {
        "version": "v4 · Summit", "agent_version": V4_AGENT_VERSION,
        "model": v4_model, "lever": "Agent Optimizer (auto)", **counts(v4_run),
        "Δ pass_rate vs v1": None, "Δ tokens vs v1": None,
    }
    last = summary.index[-1]
    summary.loc[last, "Δ pass_rate vs v1"] = round(
        summary.loc[last, "pass_rate (of graded)"] - summary.loc[0, "pass_rate (of graded)"], 2)
    summary.loc[last, "Δ tokens vs v1"] = summary.loc[last, "total_tokens"] - summary.loc[0, "total_tokens"]

    print(f"\nManual best was {PROMOTED_LABEL}. Automated optimizer produced v4 · Summit.")
    print("Compare pass_rate / tokens / elapsed_s below — did automation beat your manual climb?")
    display(summary)


Set V4_AGENT_VERSION above to the version number you applied in the portal, then re-run this cell.


## 🎉 Summary — you climbed the hill (by hand, then by machine)

You took an agent from a rough Basecamp to an auto-optimized Summit — the way real teams ship reliable AI:

- 🏕️ **Built** TrailMate v1: a grounded prompt agent (frontier model + file-search over 10 manuals).
- 🔭 **Observed** it with traces and run metrics — AgentOps, not guesswork.
- 📏 **Measured** a baseline with a **frozen** rubric + test set — your altitude.
- ⛰️ **Climbed manually, one lever at a time:** instructions (v2 Clearpath) raised quality; model (v3 Trailfinder) then cut cost/latency while holding it.
- 🏁 **Promoted** the strongest manual version as a new baseline — the manual ceiling.
- 🤖 **Automated the climb** with Agent Optimizer, which searched further than manual iteration reasonably could, producing v4 Summit.
- 🏔️ **Compared** manual vs. automated results using the exact same frozen yardstick throughout.

**The method is the point.** *Measure → change one lever → re-measure → keep what wins.* Once you've learned that discipline by hand, automation does it faster and wider — that's the real promise of Agent Optimizer.

### Where to next
- 🔁 **Optional:** let Agent Optimizer also explore *model* candidates directly (not just instructions) — a great follow-up experiment.
- 🧪 **More labs:** the `labs/more/` notebooks go deeper on multimodal chat, reasoning models, image generation, and Model Router.
- 🚀 Re-run this whole climb on your own project any time — the same code and frozen rubric make it reproducible.

**Nice work reaching the Summit — twice. 🧗🤖**
